# Crown Tracking Pipeline (Multithreshold + Consensus)

This notebook runs the complete crown tracking workflow using the reusable `TreeTrackingGraph` class in `src/flask_app_tracking/tree_tracking.py`.


In [ ]:
import importlib
import json
import os
import sys
from pathlib import Path

import geopandas as gpd

ROOT = Path.cwd().resolve().parents[1]
APP_DIR = ROOT / 'src' / 'flask_app_tracking'
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

import tree_tracking
importlib.reload(tree_tracking)
from tree_tracking import TreeTrackingGraph

print('Imported TreeTrackingGraph from:', APP_DIR)

In [ ]:
# Choose dataset: 'SIT' or 'LHC'
DATASET = 'LHC'  # <- switch to 'SIT' for SIT run
RUN_TAG = DATASET.lower()

if DATASET.upper() == 'SIT':
    MULTITHRESH_DIR = ROOT / 'output' / 'detectree_om_sit_multithreshold/crowns_multithreshold'
    ORTHO_DIR = ROOT / 'input' / 'input_om_sit'
    OUTPUT_DIR = ROOT / 'output' / f'{RUN_TAG}_tracking_rerun_1Apr26'
    SIT_OM_IDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
    VALID_LHC_STEMS = []
elif DATASET.upper() == 'LHC':
    MULTITHRESH_DIR = ROOT / 'output' / 'detectree_om_lhc_multithreshold_smaller_tiles/crowns_multithreshold'
    ORTHO_DIR = ROOT / 'input' / 'input_om_lhc'
    OUTPUT_DIR = ROOT / 'output' / f'{RUN_TAG}_tracking_rerun_1Apr26'

    # Chronological LHC sequence (excluding 9/12/25 due to gross misalignment)
    VALID_LHC_STEMS = [
        'odm_orthophoto25_10_25',
        'odm_orthophoto9_11_25',
        'odm_orthophoto20_11_25',
        'odm_orthophoto26_11_25',
        'odm_orthophoto11_1_26',
        'odm_orthophoto4_02_26',
        'odm_orthophoto20_02_26',
        'odm_orthophoto7_03_26',
    ]
    SIT_OM_IDS = []
else:
    raise ValueError(f'Unknown DATASET: {DATASET!r}. Use "SIT" or "LHC".')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATASET:', DATASET)
print('MULTITHRESH_DIR:', MULTITHRESH_DIR)
print('ORTHO_DIR:', ORTHO_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('Sequence length:', len(SIT_OM_IDS) if DATASET.upper() == 'SIT' else len(VALID_LHC_STEMS))

In [ ]:
pairs = []
missing = []

if DATASET.upper() == 'SIT':
    for om_num in SIT_OM_IDS:
        gpkg = MULTITHRESH_DIR / f'OM{om_num}_multithreshold.gpkg'
        tif = ORTHO_DIR / f'sit_om{om_num}.tif'
        stem = f'sit_om{om_num}'
        if gpkg.exists() and tif.exists():
            pairs.append((str(gpkg), str(tif), stem))
        else:
            missing.append({
                'stem': stem,
                'gpkg_exists': gpkg.exists(),
                'tif_exists': tif.exists(),
            })
else:
    for stem in VALID_LHC_STEMS:
        gpkg = MULTITHRESH_DIR / f'{stem}_multithreshold.gpkg'
        tif = ORTHO_DIR / f'{stem}.tif'
        if gpkg.exists() and tif.exists():
            pairs.append((str(gpkg), str(tif), stem))
        else:
            missing.append({
                'stem': stem,
                'gpkg_exists': gpkg.exists(),
                'tif_exists': tif.exists(),
            })

if missing:
    raise FileNotFoundError(f'Missing required files: {missing}')

tracker = TreeTrackingGraph(
    auto_discover=False,
    multithresh_dir=str(MULTITHRESH_DIR),
    ortho_dir=str(ORTHO_DIR),
    output_dir=str(OUTPUT_DIR),
    simplify_tol=1.0,
    resize_factor=0.1,
    max_crowns_preview=200,
 )

tracker.file_pairs = [(g, o) for g, o, _ in pairs]
tracker.om_ids = list(range(1, len(pairs) + 1))
tracker.base_threshold_tag = None

# Alignment strategy: SIT is stable with standard PCC; LHC can benefit from tiled PCC / ECC
ALIGN_METHOD = 'pcc_tiled'
ALIGN_THRESH_TAG = 'conf_0p65'

tracker.load_multithreshold_data(
    base_threshold_tag='conf_0p45',
    load_images=True,
    align=True,
    align_method=ALIGN_METHOD,
    align_threshold_tag=ALIGN_THRESH_TAG,
 )

print('Alignment method:', ALIGN_METHOD)
print('OM sequence loaded:')
for i, (_, _, stem) in enumerate(pairs, start=1):
    print(f'  OM{i}: {stem}')
print('Total crowns loaded:', sum(len(v) for v in tracker.crowns_gdfs.values()))

In [ ]:
# # Alignment debug summary (useful for pcc_tiled)
# import pandas as pd
# import numpy as np

# if 'tracker' not in globals():
#     raise RuntimeError('tracker not found. Run Cell 4 first.')

# dbg = getattr(tracker, 'alignment_debug', {}) or {}
# if not dbg:
#     print('No alignment_debug entries found (method may not populate debug).')
# else:
#     rows = []
#     for om_id in sorted(dbg.keys()):
#         d = dbg.get(om_id, {}) or {}
#         rej = d.get('rejected', {}) if isinstance(d.get('rejected', {}), dict) else {}
#         rows.append({
#             'om_id': int(om_id),
#             'pair': f'OM{om_id-1}->OM{om_id}',
#             'fallback': d.get('fallback', ''),
#             'n_valid_tiles': d.get('n_valid_tiles', np.nan),
#             'n_inliers': d.get('n_inliers', np.nan),
#             'median_error_inliers': d.get('median_error_inliers', d.get('median_error_all', np.nan)),
#             'rej_low_texture': rej.get('low_texture', np.nan),
#             'rej_high_error': rej.get('high_error', np.nan),
#             'rej_large_shift': rej.get('too_large_shift', np.nan),
#             'rej_exception': rej.get('exception', np.nan),
#         })

#     dbg_df = pd.DataFrame(rows).sort_values('om_id')
#     print('Alignment debug rows:', len(dbg_df))
#     print('Fallback counts:')
#     print(dbg_df['fallback'].fillna('').value_counts().to_string())
#     small_cols = [
#         'pair','n_valid_tiles','n_inliers','median_error_inliers','fallback',
#         'rej_low_texture','rej_high_error','rej_large_shift','rej_exception',
#     ]
#     print('\nPer-step debug (compact):')
#     print(dbg_df[small_cols].to_string(index=False))

In [ ]:
tracker.case_configs = tracker.make_strict_aligned_configs()

tracker.build_graph_conditional(
    base_max_dist=30.0,
    overlap_gate=0.10,
    min_base_similarity=0.30,
    classify_mode='balanced',
 )

quality_text, quality_metrics = tracker.quality_report()
print(quality_text)

tracker.save_text(quality_text, f'{RUN_TAG}_tracking_quality_report.txt')
tracker.save_json(quality_metrics, f'{RUN_TAG}_tracking_quality_metrics.json')

In [ ]:
hq = tracker.assemble_high_quality_chains()
clean_chains = hq['clean_chains']
extracted_backbones = hq['extracted_backbones']
all_extracted_chains = hq['all_extracted_chains']

# Partial-chain inclusion settings (historical SIT baseline: len>=5 and one-to-one ratio>=0.9)
MIN_PARTIAL_LEN = 5
MIN_PARTIAL_ONE_TO_ONE_RATIO = 0.9

consensus_source_chains = tracker.select_consensus_source_chains(
    hq,
    include_partial=True,
    min_partial_len=MIN_PARTIAL_LEN,
    min_partial_one_to_one_ratio=MIN_PARTIAL_ONE_TO_ONE_RATIO,
 )

print('High-quality chains:', len(all_extracted_chains))
print('  Clean chains:', len(clean_chains))
print('  Extracted backbones:', len(extracted_backbones))
print('Consensus source chains:', len(consensus_source_chains))
print('  Added filtered partial chains:', len(consensus_source_chains) - len(all_extracted_chains))
print(f'Filter settings: min_partial_len={MIN_PARTIAL_LEN}, min_partial_one_to_one_ratio={MIN_PARTIAL_ONE_TO_ONE_RATIO}')

In [ ]:
chains_viz_dir = OUTPUT_DIR / 'tracked_chains_visualization'
tracker.visualize_all_chains(all_extracted_chains, str(chains_viz_dir))
print('Saved chain visualizations to:', chains_viz_dir)


In [ ]:
# Generate consensus crowns + de-duplicate near-duplicates / (almost) containments

# Finalized tuning knobs
DEDUP_IOU_THRESHOLD = 0.75
DEDUP_CONTAINMENT_BUFFER = 5.0  # >0 expands polygons for a more tolerant containment test (units = CRS units)
DEDUP_TAG = f"iou{DEDUP_IOU_THRESHOLD:.2f}_buf{DEDUP_CONTAINMENT_BUFFER:.2f}".replace('.', 'p')

# Generate RAW consensus (1 row per source chain)
consensus_gdf_raw = tracker.generate_consensus_crowns(consensus_source_chains).reset_index(drop=True)
consensus_gdf_raw['source_chain_idx'] = list(range(len(consensus_gdf_raw)))

# De-dup at the consensus level
consensus_gdf, dedup_summary = tracker.deduplicate_crowns(
    consensus_gdf_raw,
    iou_threshold=DEDUP_IOU_THRESHOLD,
    drop_contained=True,
    containment_buffer=DEDUP_CONTAINMENT_BUFFER,
    min_area=0.0,
    verbose=False,
 )

# Filter the chain list so consensus crown visualizations match the cleaned crowns
kept_chain_indices = sorted({int(x) for x in consensus_gdf['source_chain_idx'].tolist()})
consensus_source_chains_cleaned = [consensus_source_chains[i] for i in kept_chain_indices]

# Persist outputs (cleaned is the canonical consensus artifact)
CONSENSUS_GPKG = OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg'
CONSENSUS_GPKG_RAW = OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}_raw.gpkg'
consensus_gdf.to_file(CONSENSUS_GPKG, driver='GPKG')
consensus_gdf_raw.to_file(CONSENSUS_GPKG_RAW, driver='GPKG')

summary = {
    'count_raw': int(len(consensus_gdf_raw)),
    'count_cleaned': int(len(consensus_gdf)),
    'dedup_summary': dedup_summary,
    'dedup_iou_threshold': float(DEDUP_IOU_THRESHOLD),
    'dedup_containment_buffer': float(DEDUP_CONTAINMENT_BUFFER),
    'dedup_tag': str(DEDUP_TAG),
    'consensus_gpkg': str(CONSENSUS_GPKG),
    'consensus_gpkg_raw': str(CONSENSUS_GPKG_RAW),
    'quality_counts': consensus_gdf['quality'].value_counts().to_dict() if 'quality' in consensus_gdf.columns else {},
    'num_consensus_source_chains': int(len(consensus_source_chains)),
    'num_consensus_source_chains_cleaned': int(len(consensus_source_chains_cleaned)),
}

# Canonical summary + parameter-tagged summary (for record)
tracker.save_json(summary, f'consensus_crowns_summary_{RUN_TAG}.json')
tracker.save_json(summary, f'consensus_crowns_summary_{RUN_TAG}_dedup_{DEDUP_TAG}.json')

print('Consensus crowns (raw -> cleaned):', len(consensus_gdf_raw), '->', len(consensus_gdf))
print('Consensus source chains (raw -> cleaned):', len(consensus_source_chains), '->', len(consensus_source_chains_cleaned))
print('Dedup params:', {'iou_threshold': DEDUP_IOU_THRESHOLD, 'containment_buffer': DEDUP_CONTAINMENT_BUFFER})
print('Dedup summary:', dedup_summary)
print('Saved cleaned consensus:', CONSENSUS_GPKG)
print('Saved raw consensus:', CONSENSUS_GPKG_RAW)

In [ ]:
# Export (CLEANED) consensus crowns as OM1-aligned GeoJSON
import rasterio

if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
    consensus_gpkg = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
    if not consensus_gpkg.exists():
        raise FileNotFoundError(f'Consensus file not found: {consensus_gpkg}')
    consensus_gdf = gpd.read_file(consensus_gpkg)

if consensus_gdf.empty:
    raise ValueError('Consensus GeoDataFrame is empty. Run consensus generation first.')

om1_ortho = Path(tracker.file_pairs[0][1]) if ('tracker' in globals() and tracker.file_pairs) else None
if om1_ortho is None or not om1_ortho.exists():
    raise FileNotFoundError('Could not resolve OM1 orthomosaic path from tracker.file_pairs[0].')

with rasterio.open(om1_ortho) as src:
    om1_crs = src.crs

if consensus_gdf.crs is None:
    consensus_om1 = consensus_gdf.set_crs(om1_crs, allow_override=True)
else:
    consensus_om1 = consensus_gdf.to_crs(om1_crs)

consensus_om1 = consensus_om1[
    consensus_om1.geometry.notnull() & ~consensus_om1.geometry.is_empty
] .copy()

geojson_path = OUTPUT_DIR / f'consensus_crowns_om1_phenology_{RUN_TAG}.geojson'
consensus_om1.to_file(geojson_path, driver='GeoJSON')

print('OM1 orthomosaic:', om1_ortho)
print('GeoJSON exported:', geojson_path)
print('Crown count:', len(consensus_om1))
print('Quality counts:', consensus_om1['quality'].value_counts().to_dict() if 'quality' in consensus_om1.columns else 'n/a')

In [ ]:
consensus_viz_dir = OUTPUT_DIR / 'consensus_crowns_visualization'

# Use cleaned chains (aligned with cleaned consensus crowns) when available
chains_for_viz = globals().get('consensus_source_chains_cleaned', None)
if not chains_for_viz:
    chains_for_viz = consensus_source_chains

tracker.visualize_all_consensus_chains(chains_for_viz, str(consensus_viz_dir))
print('Saved consensus visualizations to:', consensus_viz_dir)
print('Chains visualized:', len(chains_for_viz))

In [ ]:
run_summary = {
    'dataset': DATASET,
    'num_orthomosaics': len(tracker.om_ids),
    'num_graph_nodes': int(tracker.G.number_of_nodes()),
    'num_graph_edges': int(tracker.G.number_of_edges()),
    'num_high_quality_chains': int(len(all_extracted_chains)),
    'num_consensus_source_chains': int(len(consensus_source_chains)),
    'num_consensus_source_chains_cleaned': int(len(globals().get('consensus_source_chains_cleaned', []))) if 'consensus_source_chains_cleaned' in globals() else None,
    'dedup': {
        'iou_threshold': float(globals().get('DEDUP_IOU_THRESHOLD', float('nan'))),
        'containment_buffer': float(globals().get('DEDUP_CONTAINMENT_BUFFER', float('nan'))),
        'tag': str(globals().get('DEDUP_TAG', '')),
    },
    'outputs': {
        'quality_report': str(OUTPUT_DIR / f'{RUN_TAG}_tracking_quality_report.txt'),
        'quality_metrics': str(OUTPUT_DIR / f'{RUN_TAG}_tracking_quality_metrics.json'),
        'consensus_gpkg_cleaned': str(globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')),
        'consensus_gpkg_raw': str(globals().get('CONSENSUS_GPKG_RAW', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}_raw.gpkg')),
        'consensus_geojson_om1': str(OUTPUT_DIR / f'consensus_crowns_om1_phenology_{RUN_TAG}.geojson'),
        'consensus_overlay_om1': str(OUTPUT_DIR / f'consensus_overlay_om1_{RUN_TAG}.png'),
        'chain_visualizations': str(OUTPUT_DIR / 'tracked_chains_visualization'),
        'consensus_visualizations': str(OUTPUT_DIR / 'consensus_crowns_visualization'),
    },
}

print(json.dumps(run_summary, indent=2))

In [ ]:
# Crop each consensus crown geometry from every orthomosaic using alignment-corrected geometry

# This step is optional and can be expensive; keep disabled for the default end-to-end run.
RUN_CROWN_CROPS = False

import json
import imageio.v2 as imageio
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from shapely.affinity import translate
from shapely.geometry import mapping

if not RUN_CROWN_CROPS:
    print('Skipping crown crop export (set RUN_CROWN_CROPS=True to enable).')
else:
    if 'tracker' not in globals() or 'pairs' not in globals():
        raise RuntimeError('tracker/pairs not found. Run Cells 2-4 first.')

    if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
        consensus_fp = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
        if not consensus_fp.exists():
            raise FileNotFoundError(f'Consensus file not found: {consensus_fp}')
        consensus_gdf = gpd.read_file(consensus_fp)

    if consensus_gdf.empty:
        raise ValueError('Consensus crowns are empty. Run consensus generation first.')

    crop_root = OUTPUT_DIR / 'consensus_crown_crops_all_oms'
    crop_root.mkdir(parents=True, exist_ok=True)

    def to_uint8_rgb(arr):
        if arr.shape[-1] > 3:
            arr = arr[..., :3]
        if arr.shape[-1] == 1:
            arr = np.repeat(arr, 3, axis=-1)
        if arr.dtype == np.uint8:
            return arr
        arr = arr.astype(np.float32)
        lo = float(np.nanpercentile(arr, 2))
        hi = float(np.nanpercentile(arr, 98))
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            lo = float(np.nanmin(arr)) if np.isfinite(np.nanmin(arr)) else 0.0
            hi = float(np.nanmax(arr)) if np.isfinite(np.nanmax(arr)) else 1.0
            if hi <= lo:
                hi = lo + 1.0
        arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
        return (arr * 255).astype(np.uint8)

    manifest_rows = []

    for crown_idx, crown in consensus_gdf.reset_index(drop=True).iterrows():
        crown_id = f'crown_{crown_idx + 1:04d}'
        crown_dir = crop_root / crown_id
        crown_dir.mkdir(parents=True, exist_ok=True)
        geom_aligned = crown.geometry

        for om_id, (_, ortho_path, stem) in enumerate(pairs, start=1):
            dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
            geom_raw = translate(geom_aligned, xoff=-dx, yoff=-dy)

            out_png = crown_dir / f'OM{om_id:02d}_{stem}.png'
            status = 'ok'
            h = 0
            w = 0

            try:
                with rasterio.open(ortho_path) as src:
                    out_image, _ = mask(src, [mapping(geom_raw)], crop=True, filled=True, nodata=0)
                    rgb = np.moveaxis(out_image, 0, -1)
                    rgb_u8 = to_uint8_rgb(rgb)
                    h, w = rgb_u8.shape[:2]
                    imageio.imwrite(out_png, rgb_u8)
            except ValueError:
                status = 'no_overlap'
            except Exception as e:
                status = f'error: {str(e)}'

            manifest_rows.append({
                'crown_id': crown_id,
                'consensus_index': int(crown_idx),
                'om_id': int(om_id),
                'stem': stem,
                'shift_dx_used': float(dx),
                'shift_dy_used': float(dy),
                'status': status,
                'height_px': int(h),
                'width_px': int(w),
                'crop_path': str(out_png),
            })

    manifest_df = pd.DataFrame(manifest_rows)
    manifest_csv = crop_root / 'consensus_crops_manifest.csv'
    manifest_json = crop_root / 'consensus_crops_manifest.json'
    manifest_df.to_csv(manifest_csv, index=False)
    with open(manifest_json, 'w') as f:
        json.dump(manifest_rows, f, indent=2)

    summary = {
        'dataset': DATASET,
        'num_consensus_crowns': int(consensus_gdf.shape[0]),
        'num_orthomosaics': int(len(pairs)),
        'expected_total_crops': int(consensus_gdf.shape[0] * len(pairs)),
        'successful_crops': int((manifest_df['status'] == 'ok').sum()),
        'failed_or_missing': int((manifest_df['status'] != 'ok').sum()),
        'manifest_csv': str(manifest_csv),
        'manifest_json': str(manifest_json),
        'output_dir': str(crop_root),
    }

    print(json.dumps(summary, indent=2))

In [ ]:
# # Build one time-series image per crown (OM1..OM8 panels in a single figure)
# import math
# import numpy as np
# import imageio.v2 as imageio
# import matplotlib.pyplot as plt
# from pathlib import Path
# from IPython.display import Image, display

# if 'manifest_df' not in globals() or manifest_df.empty:
#     manifest_path = OUTPUT_DIR / 'consensus_crown_crops_all_oms' / 'consensus_crops_manifest.csv'
#     if not manifest_path.exists():
#         raise FileNotFoundError(f'Manifest not found: {manifest_path}')
#     manifest_df = pd.read_csv(manifest_path)

# panel_root = OUTPUT_DIR / 'consensus_crown_timeseries_panels'
# panel_root.mkdir(parents=True, exist_ok=True)

# good = manifest_df[manifest_df['status'] == 'ok'].copy()
# if good.empty:
#     raise RuntimeError('No successful crops found in manifest.')

# # Keep OM order consistent
# good = good.sort_values(['crown_id', 'om_id']).reset_index(drop=True)

# panel_records = []
# for crown_id, sub in good.groupby('crown_id', sort=True):
#     sub = sub.sort_values('om_id')
#     n = len(sub)
#     fig_w = max(16, 2.8 * n)
#     fig, axes = plt.subplots(1, n, figsize=(fig_w, 3.6))
#     if n == 1:
#         axes = [axes]

#     for ax, row in zip(axes, sub.itertuples(index=False)):
#         img_path = Path(row.crop_path)
#         try:
#             img = imageio.imread(img_path)
#             ax.imshow(img)
#             ax.set_title(f'OM{int(row.om_id)}\n{row.stem}', fontsize=9)
#         except Exception:
#             ax.text(0.5, 0.5, 'read error', ha='center', va='center')
#             ax.set_title(f'OM{int(row.om_id)}\n{row.stem}', fontsize=9)
#         ax.axis('off')

#     fig.suptitle(f'{crown_id} time series', fontsize=12, y=1.04)
#     fig.tight_layout()
#     out_path = panel_root / f'{crown_id}_timeseries.png'
#     fig.savefig(out_path, dpi=160, bbox_inches='tight')
#     plt.close(fig)
#     panel_records.append({'crown_id': crown_id, 'panel_path': str(out_path), 'num_oms': int(n)})

# panel_df = pd.DataFrame(panel_records)
# panel_csv = panel_root / 'timeseries_panels_manifest.csv'
# panel_df.to_csv(panel_csv, index=False)

# print(f'Time-series panels generated: {len(panel_df)}')
# print(f'Output folder: {panel_root}')
# print(f'Manifest: {panel_csv}')

# # Preview first 3 panels
# for p in panel_df['panel_path'].head(3):
#     print(Path(p).name)
#     display(Image(filename=p, width=1800))

In [ ]:
# Save spatial overlay (CLEANED): consensus crowns over OM1
# Writes a PNG; does not display.

import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio
from pathlib import Path

plt.ioff()

if 'tracker' not in globals() or tracker is None:
    raise RuntimeError('tracker not found. Run Cells 2–5 first.')

# Load cleaned consensus crowns if not already in memory
if 'consensus_gdf' not in globals() or consensus_gdf is None or len(consensus_gdf) == 0:
    consensus_path = globals().get('CONSENSUS_GPKG', OUTPUT_DIR / f'consensus_crowns_complete_all_{RUN_TAG}.gpkg')
    if not Path(consensus_path).exists():
        raise FileNotFoundError(f'Consensus file not found: {consensus_path}')
    consensus_gdf = gpd.read_file(consensus_path)

if consensus_gdf.empty:
    raise ValueError('Consensus crowns are empty. Nothing to overlay.')

om1_path = Path(tracker.file_pairs[0][1]) if tracker.file_pairs and tracker.file_pairs[0][1] else None
if om1_path is None or not om1_path.exists():
    raise FileNotFoundError('Could not resolve OM1 orthomosaic path (tracker.file_pairs[0][1]).')

# Read a downsampled RGB for fast visualization
MAX_PREVIEW = 2400
with rasterio.open(om1_path) as src:
    scale = max(src.width, src.height) / MAX_PREVIEW if max(src.width, src.height) > MAX_PREVIEW else 1.0
    out_w = int(round(src.width / scale))
    out_h = int(round(src.height / scale))
    data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
    img = np.moveaxis(data, 0, -1).astype(np.float32)
    # Robust contrast stretch
    lo = np.nanpercentile(img, 2)
    hi = np.nanpercentile(img, 98)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo = float(np.nanmin(img)) if np.isfinite(np.nanmin(img)) else 0.0
        hi = float(np.nanmax(img)) if np.isfinite(np.nanmax(img)) else 1.0
        if hi <= lo:
            hi = lo + 1.0
    img = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
    tr = src.transform * rasterio.transform.Affine.scale(src.width / out_w, src.height / out_h)
    # Extent for imshow in map coordinates
    left, top = (tr.c, tr.f)
    right = left + out_w * tr.a
    bottom = top + out_h * tr.e
    extent = [left, right, bottom, top]
    om1_crs = src.crs

overlay = consensus_gdf.copy()
if overlay.crs is None:
    overlay = overlay.set_crs(om1_crs, allow_override=True)
elif om1_crs is not None and overlay.crs != om1_crs:
    overlay = overlay.to_crs(om1_crs)

overlay = overlay[overlay.geometry.notnull() & ~overlay.geometry.is_empty].copy()
if overlay.empty:
    raise ValueError('Consensus crowns have no valid geometries after filtering.')

fig, ax = plt.subplots(figsize=(14, 14))
ax.imshow(img, extent=extent)
overlay.boundary.plot(ax=ax, linewidth=1.2, color='yellow', alpha=0.95)
ax.set_title(f'Consensus crowns over OM1 (CLEANED) ({DATASET}, n={len(overlay)})')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_aspect('equal', adjustable='box')

out_png = OUTPUT_DIR / f'consensus_overlay_om1_{RUN_TAG}.png'
fig.savefig(out_png, dpi=220, bbox_inches='tight')
plt.close(fig)

print('Saved cleaned overlay PNG:', out_png)